# Experiment 2: Android Runtime Deployment and Validation

## Objective

The objective of this experiment is to deploy the Snapdragon-enabled `llama.cpp` runtime to the target Android device and validate that the deployment environment is configured correctly.

Specifically, this experiment verifies:

- Deployment of the generated package
- Executable accessibility
- Runtime environment configuration
- Initial runtime invocation

The experiment intentionally concludes at the first reproducible runtime failure.

Detailed dependency investigation and root cause analysis are deferred to **Experiment 03: Runtime Dependency Analysis**.

## Background

Experiment 1 successfully produced a Snapdragon-enabled build of `llama.cpp`.

Generated artifacts include:

- `llama-cli`
- `llama-server`
- `llama-bench`
- Qualcomm backend libraries

The next step is to deploy these artifacts to the Android device and verify that the runtime initializes correctly.

## Runtime Architecture

The runtime execution flow is shown below.

```text
llama-cli
      │
      ▼
libggml.so
      │
      ▼
libggml-hexagon.so
      │
      ▼
Qualcomm AI Runtime
      │
      ▼
Hexagon Driver
      │
      ▼
HTP Devices
```

This diagram illustrates the expected runtime execution path.

This experiment validates deployment and attempts runtime initialization up to the first observable failure.

## Experiment Goal

The experiment is considered successful if the following objectives are achieved:

- Deployment package transferred successfully
- Executables are accessible on the target device
- Runtime library paths are configured correctly
- Runtime initialization is attempted
- The first runtime failure (if any) is reproduced

Detailed dependency investigation is intentionally outside the scope of this experiment.

### Step 1: Verify Android Device

```bash
adb devices
```

### Step 2: Create Deployment Directory

```bash
adb shell "mkdir -p /data/local/tmp/llama"
```

### Step 3: Transfer Package

```bash
adb push llama.cpp/pkg-snapdragon/llama.cpp/. /data/local/tmp/llama/
```

### Step 4: Verify Deployment

```bash
adb shell "ls -R /data/local/tmp/llama"
```

## Configure Runtime

Configure the runtime library paths before executing the binaries.

```bash
adb shell
```

Then inside:
```bash
cd /data/local/tmp/llama

export LD_LIBRARY_PATH=$PWD/lib:/vendor/lib64:/system/lib64
export ADSP_LIBRARY_PATH=/vendor/lib/rfsa/adsp:/system/lib/rfsa/adsp:$PWD/lib
```

## Validate Runtime

```bash
./bin/llama-cli --help
```

## Expected Outcome

The executable should begin runtime initialization after the environment variables are configured.

If runtime initialization fails, the first reproducible failure will be recorded as the outcome of this experiment.

Root cause analysis is continued in **Experiment 03: Runtime Dependency Analysis**.

## Observation

### Observed Output
```bash
CANNOT LINK EXECUTABLE "./bin/llama-cli":
cannot locate symbol "_ZNSt3__113__hash_memoryEPKvm"
referenced by "/system/lib64/liblog.so"
```

The executable was successfully launched by Android's runtime loader.

However, execution terminated during the dynamic linking stage before the application reached backend initialization.

The failure is deterministic and reproducible across multiple executions.

This concludes the runtime validation experiment.

Further investigation into the dependency chain and linker failure is continued in **Experiment 03: Runtime Dependency Analysis**.

## Conclusion

The deployment pipeline was successfully validated.

The following stages completed successfully:

- Snapdragon package generation
- Android deployment using ADB
- Runtime permission configuration
- Runtime environment configuration
- Executable invocation

Execution terminated during Android's dynamic linking stage with a reproducible linker error.

This establishes a stable baseline for **Experiment 03**, which investigates the runtime dependency chain and linker failure.

## Outcome

| Stage | Status |
|--------|--------|
| Snapdragon Backend Build | ✅ Completed |
| Package Generation | ✅ Completed |
| Device Deployment | ✅ Completed |
| Runtime Configuration | ✅ Completed |
| Executable Launch | ✅ Completed |
| Android Linker Initialization | ✅ Reached |
| Backend Initialization | ❌ Not Reached |
| Model Loading | ⏸ Pending |

The runtime validation experiment concludes here.

The remaining linker failure becomes the starting point for **Experiment 03**.